## Preprocessing RightWhalecalls

### Membaca file .ts

In [2]:
def read_ts_file(file_path):
    data = []
    labels = []
    is_data_section = False

    with open(file_path, "r") as f:
        for line in f:
            line = line.strip()

            # Mulai membaca setelah @data
            if line.lower() == "@data":
                is_data_section = True
                continue

            if not is_data_section:
                continue

            # Pisahkan nilai dan label
            if ":" in line:
                values, label = line.split(":")
                series = [float(x) for x in values.split(",") if x != ""]
                data.append(series)
                labels.append(label)

    return data, labels


### Load File Train dan Test

In [3]:
train_path = "RightWhaleCalls/RightWhaleCalls_TRAIN.ts"
test_path  = "RightWhaleCalls/RightWhaleCalls_TEST.ts"

X_train, y_train = read_ts_file(train_path)
X_test, y_test = read_ts_file(test_path)

print("File loaded successfully!")


File loaded successfully!


### Konversi Data ke NumPy Array

In [4]:
import numpy as np

X_train_np = np.array(X_train)
X_test_np  = np.array(X_test)

y_train_np = np.array(y_train)
y_test_np  = np.array(y_test)

print("Shape X_train:", X_train_np.shape)
print("Shape X_test :", X_test_np.shape)


Shape X_train: (9324, 4000)
Shape X_test : (1962, 4000)


### Cek Panjang Time Series (Validasi Data)

In [5]:
train_lengths = [len(ts) for ts in X_train_np]
test_lengths  = [len(ts) for ts in X_test_np]

print("TRAIN - unique lengths:", set(train_lengths))
print("TEST  - unique lengths:", set(test_lengths))


TRAIN - unique lengths: {4000}
TEST  - unique lengths: {4000}


### Normalisasi Data (Z-Score)

In [6]:
def z_normalize(data):
    mean = np.mean(data, axis=1, keepdims=True)
    std  = np.std(data, axis=1, keepdims=True)
    return (data - mean) / (std + 1e-8)

X_train_norm = z_normalize(X_train_np)
X_test_norm  = z_normalize(X_test_np)

print("Normalisasi selesai")


Normalisasi selesai


### Encoding Label (String → Numerik)

In [7]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_enc = le.fit_transform(y_train_np)
y_test_enc  = le.transform(y_test_np)

print("Mapping label:")
for i, cls in enumerate(le.classes_):
    print(f"{cls} -> {i}")


Mapping label:
0 -> 0
1 -> 1


### Reshape Data (untuk Algoritma CNN)

In [8]:
X_train_cnn = X_train_norm[..., np.newaxis]
X_test_cnn  = X_test_norm[..., np.newaxis]

print("Shape untuk CNN:")
print("X_train:", X_train_cnn.shape)
print("X_test :", X_test_cnn.shape)


Shape untuk CNN:
X_train: (9324, 4000, 1)
X_test : (1962, 4000, 1)


### Cek Missing / Invalid Value

In [ ]:
print("NaN di X_train:", np.isnan(X_train_norm).sum())
print("NaN di X_test :", np.isnan(X_test_norm).sum())


NaN di X_train: 0
NaN di X_test : 0


: 